In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import (
    SelectKBest,
    mutual_info_regression,
    f_regression
)
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import mlflow
import mlflow.sklearn

import warnings

warnings.filterwarnings("ignore")

In [2]:
DATA_PATH = "../data/processed/CIB_features.csv"

df = pd.read_csv(DATA_PATH)

df["Date"] = pd.to_datetime(df["Date"])

df = df.sort_values("Date").reset_index(drop=True)

print("Shape:", df.shape)

df.head()

Shape: (1265, 34)


,Date,Close,Open,High,Low,Volume,Change %,Return_1d,Return_3d,Return_5d,...,High_Low_Range,Open_Close_Range,Volume_Change,Volume_SMA_10,Close_lag_1,Close_lag_2,Close_lag_3,Close_lag_5,Close_lag_10,Target
0,2021-09-01,1.855,1.855,1.862,1.771,65200.0,3.02,NaN,NaN,NaN,...,0.049057,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.849
1,2021-09-02,1.849,1.849,1.849,1.801,177910.0,-0.33,-0.003235,NaN,NaN,...,0.025960,0.000000,1.728681,NaN,1.855,NaN,NaN,NaN,NaN,1.843
2,2021-09-03,1.843,1.836,1.843,1.775,69020.0,-0.33,-0.003245,NaN,NaN,...,0.036896,0.003813,-0.612051,NaN,1.849,1.855,NaN,NaN,NaN,1.795
3,2021-09-07,1.795,1.749,1.801,1.749,83090.0,-2.62,-0.026044,-0.032345,NaN,...,0.028969,0.026301,0.203854,NaN,1.843,1.849,1.855,NaN,NaN,1.819
4,2021-09-08,1.819,1.801,1.819,1.777,29790.0,1.35,0.013370,-0.016225,NaN,...,0.023090,0.009994,-0.641473,NaN,1.795,1.843,1.849,NaN,NaN,1.795


In [4]:
print("Start date:", df["Date"].min())
print("End date:", df["Date"].max())



Start date: 2021-09-01 00:00:00
End date: 2026-09-30 00:00:00


In [5]:
# =========================
# Time-Based Split
# =========================

train = df[df["Date"] < "2025-01-01"].copy()

valid = df[
    (df["Date"] >= "2025-01-01") &
    (df["Date"] < "2026-01-01")
].copy()

test = df[
    (df["Date"] >= "2026-01-01") &
    (df["Date"] < "2026-07-01")
].copy()

tracking = df[
    df["Date"] >= "2026-07-01"
].copy()

In [6]:
print("Train:", train.shape)
print("Validation:", valid.shape)
print("Test:", test.shape)
print("Tracking:", tracking.shape)

Train: (837, 34)
Validation: (243, 34)
Test: (122, 34)
Tracking: (63, 34)


In [7]:
print("\nTrain:")
print(train["Date"].min(), "→", train["Date"].max())

print("\nValidation:")
print(valid["Date"].min(), "→", valid["Date"].max())

print("\nTest:")
print(test["Date"].min(), "→", test["Date"].max())

print("\nTracking:")
print(tracking["Date"].min(), "→", tracking["Date"].max())


Train:
2021-09-01 00:00:00 → 2024-12-31 00:00:00

Validation:
2025-01-02 00:00:00 → 2025-12-31 00:00:00

Test:
2026-01-02 00:00:00 → 2026-06-30 00:00:00

Tracking:
2026-07-01 00:00:00 → 2026-09-30 00:00:00


In [8]:
df.columns

Index(['Date', 'Close', 'Open', 'High', 'Low', 'Volume', 'Change %',
       'Return_1d', 'Return_3d', 'Return_5d', 'Return_10d', 'SMA_10', 'SMA_20',
       'EMA_10', 'EMA_20', 'RSI_14', 'MACD', 'MACD_Signal', 'MACD_Hist',
       'BB_Upper', 'BB_Lower', 'BB_Width', 'Volatility_10', 'Volatility_20',
       'High_Low_Range', 'Open_Close_Range', 'Volume_Change', 'Volume_SMA_10',
       'Close_lag_1', 'Close_lag_2', 'Close_lag_3', 'Close_lag_5',
       'Close_lag_10', 'Target'],
      dtype='object')

# feature selection

In [9]:
target = "Target"
feature_cols = [
    "Open",
    "High",
    "Low",
    "Close",
    "Volume",
    "Return_1d",
    "Return_3d",
    "Return_5d",
    "Return_10d",
    "SMA_10",
    "SMA_20",
    "EMA_10",
    "EMA_20",
    "RSI_14",
    "MACD",
    "MACD_Signal",
    "MACD_Hist",
    "BB_Width",
    "Volatility_10",
    "Volatility_20",
    "High_Low_Range",
    "Open_Close_Range",
    "Volume_Change",
    "Volume_SMA_10",
    "Close_lag_1",
    "Close_lag_2",
    "Close_lag_3",
    "Close_lag_5",
    "Close_lag_10"
]

In [10]:
X_train = train[feature_cols].copy()
y_train = train[target].copy()

X_valid = valid[feature_cols].copy()
y_valid = valid[target].copy()

X_test = test[feature_cols].copy()
y_test = test[target].copy()

X_tracking = tracking[feature_cols].copy()
y_tracking = tracking[target].copy()

In [11]:
print("Train NaNs:", X_train.isnull().sum().sum())
print("Validation NaNs:", X_valid.isnull().sum().sum())
print("Test NaNs:", X_test.isnull().sum().sum())
print("Tracking NaNs:", X_tracking.isnull().sum().sum())

print("\nTarget NaNs:")
print("Train:", y_train.isnull().sum())
print("Validation:", y_valid.isnull().sum())
print("Test:", y_test.isnull().sum())
print("Tracking:", y_tracking.isnull().sum())

Train NaNs: 141
Validation NaNs: 0
Test NaNs: 0
Tracking NaNs: 0

Target NaNs:
Train: 0
Validation: 0
Test: 0
Tracking: 1


In [12]:
train_mask = X_train.notna().all(axis=1)

X_train = X_train.loc[train_mask]
y_train = y_train.loc[train_mask]

print("Train shape after removing NaNs:", X_train.shape)
print("Train NaNs:", X_train.isnull().sum().sum())
print("Train target NaNs:", y_train.isnull().sum())

Train shape after removing NaNs: (817, 29)
Train NaNs: 0
Train target NaNs: 0


In [13]:
target_corr = pd.concat(
    [X_train, y_train.rename("Target")],
    axis=1
).corr()["Target"].drop("Target").sort_values(ascending=False)

target_corr

Close               0.992747
Low                 0.991234
High                0.990034
Open                0.988054
Close_lag_1         0.986843
Close_lag_2         0.981531
EMA_10              0.979053
Close_lag_3         0.975734
SMA_10              0.974824
Close_lag_5         0.962935
EMA_20              0.958827
SMA_20              0.944671
Close_lag_10        0.924165
MACD_Signal         0.307818
MACD                0.299841
RSI_14              0.213909
Return_10d          0.159389
Return_5d           0.101381
Return_3d           0.075534
Return_1d           0.043067
MACD_Hist           0.031204
BB_Width            0.002485
Volume_Change      -0.031196
Open_Close_Range   -0.039754
Volatility_10      -0.088750
Volatility_20      -0.102038
Volume             -0.121847
Volume_SMA_10      -0.170147
High_Low_Range     -0.237809
Name: Target, dtype: float64

## # Linear Regression

In [15]:
features = [
    "Open",
    "High",
    "Low",
    "Close"
]

X_train_ohlc = X_train[features]
X_valid_ohlc = X_valid[features]
X_test_ohlc = X_test[features]

In [16]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", LinearRegression())
])

model.fit(X_train_ohlc, y_train)

y_pred_valid = model.predict(X_valid_ohlc)
y_pred_test = model.predict(X_test_ohlc)

In [17]:
def evaluate_model(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2 = r2_score(y_true, y_pred)

    return mae, rmse, r2


valid_metrics = evaluate_model(y_valid, y_pred_valid)
test_metrics = evaluate_model(y_test, y_pred_test)

print("Validation:")
print("MAE:", valid_metrics[0])
print("RMSE:", valid_metrics[1])
print("R²:", valid_metrics[2])

print("\nTest:")
print("MAE:", test_metrics[0])
print("RMSE:", test_metrics[1])
print("R²:", test_metrics[2])

Validation:
MAE: 0.021210504349652957
RMSE: 0.028761084328100504
R²: 0.9893158479130242

Test:
MAE: 0.050851660047846685
RMSE: 0.06806693818834546
R²: 0.863140429643104


## experiment to select features

In [18]:
features = [
    "Open",
    "High",
    "Low",
    "Close",
    "Close_lag_1",
    "EMA_10",
    "SMA_10"
]

X_train_exp = X_train[features]
X_valid_exp = X_valid[features]
X_test_exp = X_test[features]

In [19]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", LinearRegression())
])

model.fit(X_train_exp, y_train)

y_pred_valid = model.predict(X_valid_exp)
y_pred_test = model.predict(X_test_exp)

In [20]:
valid_metrics = evaluate_model(y_valid, y_pred_valid)
test_metrics = evaluate_model(y_test, y_pred_test)

print("Validation:")
print("MAE:", valid_metrics[0])
print("RMSE:", valid_metrics[1])
print("R²:", valid_metrics[2])

print("\nTest:")
print("MAE:", test_metrics[0])
print("RMSE:", test_metrics[1])
print("R²:", test_metrics[2])

Validation:
MAE: 0.02114942583522717
RMSE: 0.028909097012849272
R²: 0.9892055976069477

Test:
MAE: 0.05099204202626511
RMSE: 0.06844731979200634
R²: 0.8616065182991279


In [21]:
with mlflow.start_run(run_name="Linear_Regression_OHLC_Lag_MA"):

    mlflow.log_param("model", "Linear Regression")
    mlflow.log_param(
        "features",
        "Open,High,Low,Close,Close_lag_1,EMA_10,SMA_10"
    )
    mlflow.log_param("features_count", len(features))

    mlflow.log_metric("val_mae", valid_metrics[0])
    mlflow.log_metric("val_rmse", valid_metrics[1])
    mlflow.log_metric("val_r2", valid_metrics[2])

    mlflow.log_metric("test_mae", test_metrics[0])
    mlflow.log_metric("test_rmse", test_metrics[1])
    mlflow.log_metric("test_r2", test_metrics[2])

    mlflow.sklearn.log_model(model, "model")

2026/10/07 21:16:59 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
